# Part B — flatten the JSON event log to columnar Parquet

Pure Python (pyarrow + pandas) — no JSONiq/Spark here. We define a target schema (B1), a
row-flattening function (B2), stream the ~70 GB JSONL file into `events.parquet` in 10k-row
row groups (B3), then validate the output (B4).

Note: the assignment names the source `git-archive-big.json.gz`; the file actually present is
`data/git-archive-huge.json` (uncompressed). The reader below handles both (`.gz` or plain).

## B1 — schema definition

A `pa.schema` matching the required columns. `event_type` is dictionary-encoded (small int8
codes + a string dictionary) because it has only ~14 distinct values; `created_at` is a
microsecond UTC timestamp; `commit_count` is a 32-bit int.

In [1]:
import pyarrow as pa
import pyarrow.parquet as pq

SCHEMA = pa.schema([
    ("event_id",     pa.string()),
    ("event_type",   pa.dictionary(pa.int8(), pa.string())),
    ("actor_login",  pa.string()),
    ("repo_name",    pa.string()),
    ("created_at",   pa.timestamp("us", tz="UTC")),
    ("commit_count", pa.int32()),
])

# Output of this step is the printed schema.
print(SCHEMA)

event_id: string
event_type: dictionary<values=string, indices=int8, ordered=0>
actor_login: string
repo_name: string
created_at: timestamp[us, tz=UTC]
commit_count: int32


## B2 — `flatten_event(raw) -> dict`

Turns one raw event object into a flat row. Handles the documented edge cases for the commit
count: **payload missing**, **payload `{}`**, or **`payload.size == 0`** all yield `0`.
`actor`/`repo` are guarded with `.get(...)` too, so a rare event missing those fields can't
abort the multi-million-row conversion.

In [2]:
def flatten_event(raw: dict) -> dict:
    payload = raw.get("payload") or {}        # missing OR {} -> {}
    actor   = raw.get("actor") or {}
    repo    = raw.get("repo") or {}
    return {
        "event_id":     raw.get("id"),
        "event_type":   raw.get("type"),
        "actor_login":  actor.get("login"),
        "repo_name":    repo.get("name"),
        "created_at":   raw.get("created_at"),     # ISO-8601 string; cast to timestamp in B3
        "commit_count": int(payload.get("size") or 0),   # 0 if missing/empty/size 0
    }

# Sanity-check the three documented edge cases (each must give commit_count == 0):
base = {"id": "x", "type": "PushEvent", "actor": {"login": "a"},
        "repo": {"name": "r"}, "created_at": "2015-01-01T00:00:00Z"}
assert flatten_event({**base})["commit_count"] == 0                      # payload missing
assert flatten_event({**base, "payload": {}})["commit_count"] == 0       # payload {}
assert flatten_event({**base, "payload": {"size": 0}})["commit_count"] == 0  # size 0
assert flatten_event({**base, "payload": {"size": 3}})["commit_count"] == 3  # normal
print("flatten_event OK: missing / {} / size-0 payload all -> commit_count 0")

flatten_event OK: missing / {} / size-0 payload all -> commit_count 0


## B3 — stream JSON → Parquet in 10,000-row batches

Read the file line by line (constant memory), accumulate 10k flattened rows, convert each
batch to a `pa.Table` (casting to `SCHEMA`), and write it as one **row group** via
`ParquetWriter.write_table()`. Malformed lines are skipped (counted), so a single bad line
can't abort the whole run. This is the slow step — single-threaded over ~28.5M events.

In [3]:
import json, gzip, time, os

SRC = "C:/Users/daniel pilant/folders/study_material/Year_3_Semester_2/BigData/Week6/data/git-archive-huge.json"
OUT = "events.parquet"
BATCH = 10_000

def open_text(path):
    """Open .gz transparently, else a plain text file."""
    if path.endswith(".gz"):
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "rt", encoding="utf-8")

def flush(batch, writer):
    """Convert a batch of dicts to a schema-typed table and write it as a row group."""
    table = pa.Table.from_pylist(batch).cast(SCHEMA)
    if writer is None:
        writer = pq.ParquetWriter(OUT, SCHEMA, compression="snappy")
    writer.write_table(table)
    return writer

rows = bad = 0
writer = None
batch = []
start = time.perf_counter()
try:
    with open_text(SRC) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                batch.append(flatten_event(json.loads(line)))
            except Exception:
                bad += 1
                continue
            if len(batch) >= BATCH:
                writer = flush(batch, writer)
                rows += len(batch)
                batch.clear()
    if batch:                       # final partial batch
        writer = flush(batch, writer)
        rows += len(batch)
finally:
    if writer is not None:
        writer.close()

elapsed_b3 = time.perf_counter() - start
print(f"Wrote {rows:,} events to {OUT}  (skipped {bad} malformed lines)")
print(f"B3 wall-clock: {elapsed_b3:,.1f} s   throughput: {rows/elapsed_b3:,.0f} rows/s")

Wrote 28,506,909 events to events.parquet  (skipped 0 malformed lines)
B3 wall-clock: 406.8 s   throughput: 70,074 rows/s


## B4 — validate the output

Reload `events.parquet` into a DataFrame (timed = B4) and print: (a) the file schema,
(b) the first 500 rows, (c) parquet vs source size and the ratio, (d) the DataFrame's
in-memory footprint.

In [4]:
import pandas as pd

# (timed) reload the whole Parquet file into a pandas DataFrame
start = time.perf_counter()
df = pd.read_parquet(OUT)
elapsed_b4 = time.perf_counter() - start

# a. file schema
print("a) Parquet schema:")
print(pq.read_schema(OUT))

# c. sizes and ratio  (source here is the uncompressed .json, not a .gz)
parquet_mb = os.path.getsize(OUT) / 1024**2
source_mb  = os.path.getsize(SRC) / 1024**2
print(f"\nc) events.parquet : {parquet_mb:,.1f} MB")
print(f"   source JSON     : {source_mb:,.1f} MB")
print(f"   ratio (source / parquet): {source_mb/parquet_mb:,.1f}x smaller")

# d. DataFrame memory footprint
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f"\nd) DataFrame memory: {mem_mb:,.1f} MB  ({len(df):,} rows)")
print(f"\nB4 reload wall-clock: {elapsed_b4:,.1f} s   throughput: {len(df)/elapsed_b4:,.0f} rows/s")

# b. first 500 rows
print("\nb) first 500 rows:")
df.head(500)

a) Parquet schema:


event_id: string
event_type: dictionary<values=string, indices=int8, ordered=0>
actor_login: string
repo_name: string
created_at: timestamp[us, tz=UTC]
commit_count: int32

c) events.parquet : 829.7 MB
   source JSON     : 71,528.4 MB
   ratio (source / parquet): 86.2x smaller

d) DataFrame memory: 2,109.4 MB  (28,506,909 rows)

B4 reload wall-clock: 12.9 s   throughput: 2,208,199 rows/s

b) first 500 rows:


,event_id,event_type,actor_login,repo_name,created_at,commit_count
0,2594237218,IssueCommentEvent,yongli-d,yongli-d/StravaBuddy,2015-02-20 01:00:00+00:00,0
1,2594237220,CreateEvent,vgeshel,yummly/s3-to-redshift,2015-02-20 01:00:01+00:00,0
2,2594237222,PushEvent,davidcarlsonberg,PubWlkr/PubWlkr,2015-02-20 01:00:01+00:00,1
3,2594237223,IssueCommentEvent,wronk,mne-tools/mne-python,2015-02-20 01:00:01+00:00,0
4,2594237233,WatchEvent,mcstiches,raywenderlich/swift-style-guide,2015-02-20 01:00:01+00:00,0
...,...,...,...,...,...,...
495,2594239859,PushEvent,TomPedersen,TomPedersen/TomPedersen.github.io,2015-02-20 01:01:23+00:00,1
496,2594239861,DeleteEvent,ekmartin,webkom/vote,2015-02-20 01:01:23+00:00,0
497,2594239877,PushEvent,mcgriffin,mcgriffin/Resume,2015-02-20 01:01:23+00:00,1
498,2594239882,PushEvent,Naios,Naios/TrinityCore,2015-02-20 01:01:24+00:00,1
